In [1]:
import numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

daily = pd.read_csv('../data/daily_sales.csv', parse_dates=['Date'])
values = daily['Revenue'].values.reshape(-1,1)

scaler = MinMaxScaler()
scaled = scaler.fit_transform(values)

# Create sequences
SEQ_LEN = 14
def make_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data)-seq_len):
        X.append(data[i:i+seq_len])
        y.append(data[i+seq_len])
    return np.array(X), np.array(y)

X, y = make_sequences(scaled, SEQ_LEN)
split = int(len(X)*0.8)
X_train, X_test = torch.tensor(X[:split], dtype=torch.float32), torch.tensor(X[split:], dtype=torch.float32)
y_train, y_test = torch.tensor(y[:split], dtype=torch.float32), torch.tensor(y[split:], dtype=torch.float32)

In [3]:
class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=64, num_layers=2, batch_first=True)
        self.fc   = nn.Linear(64, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

model = LSTMModel()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

for epoch in range(30):
    model.train()
    pred = model(X_train)
    loss = loss_fn(pred, y_train)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    if epoch % 10 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item():.4f}')

Epoch 0, Loss: 0.0135
Epoch 10, Loss: 0.0061
Epoch 20, Loss: 0.0058


In [4]:
# Evaluate
model.eval()
with torch.no_grad():
    preds = scaler.inverse_transform(model(X_test).numpy())
    actuals = scaler.inverse_transform(y_test.numpy())

from sklearn.metrics import mean_absolute_percentage_error
mape = mean_absolute_percentage_error(actuals, preds)
print(f'LSTM MAPE: {mape*100:.2f}%')

torch.save(model.state_dict(), '../models/lstm_model.pth')

LSTM MAPE: 41.85%
